<a href="https://colab.research.google.com/github/Castlebin/Hands-On-Large-Language-Models-CN/blob/my_master/0_my_code/ch01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [2]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

In [1]:
# 要安装的包
!pip install transformers accelerate vllm
# 安装我自己的包
!pip install dl_d2l matplotlib_cn

In [2]:
import os
from dl_d2l.util import colab_util

# 缓存目录
base_data_dir = colab_util.get_base_data_dir()
print(f"base data dir: {base_data_dir} \n")

# 数据集缓存目录
datasets_dir = os.path.join(base_data_dir, "ML", "Datasets")
os.makedirs(datasets_dir, exist_ok=True)
print(f"datasets dir: {datasets_dir} \n")

# 让 matplotlib 绘图 支持中文显示
from matplotlib_cn import matplotlib_util
matplotlib_util.enable_chinese()

# 使用 AutoModelForCausalLM.from_pretrained() 下载模型时，可以通过 cache_dir 指定缓存目录，下面将使用到
# huggingface 缓存目录
hf_cache_dir = os.path.join(base_data_dir, "ML", "huggingface")
print(f"huggingface cache_dir: {hf_cache_dir} \n")

Current environment is Google Colab, mounting Google Drive...
Google Drive data directory ready: /content/drive/MyDrive/data
base data dir: /content/drive/MyDrive/data 

datasets dir: /content/drive/MyDrive/data/ML/Datasets 

huggingface cache_dir: /content/drive/MyDrive/data/ML/huggingface 



# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).


使用 transformers 库，加载模型和它的 tokenizer 。这里使用 Microsoft 的 Phi-3

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
## 加载模型
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
    cache_dir=hf_cache_dir    # 指定缓存目录
)
# 加载模型的 tokenizer (Embedding 模型)
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    cache_dir=hf_cache_dir    # 指定缓存目录
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [4]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Finally, we create our prompt as a user and give it to the model:

In [5]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])

 Why did the chicken join the band? Because it had the drumsticks!


# -----  英文原版内容结束 --------
下面是扩展内容

## 使用国产大模型 Qwen

按同样的方式，来尝试一下国产大模型 Qwen



In [7]:
# 使用 国产大模型千问 Qwen/Qwen2.5-0.5B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

## 1. 加载模型 和 它的 tokenizer
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype="auto",
    trust_remote_code=False,
    cache_dir=hf_cache_dir    # 指定缓存目录
)
## 2. 加载 tokenizer
tokenizer_qwen = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=hf_cache_dir    # 指定缓存目录
)

In [8]:
from transformers import pipeline

# Create a pipeline
generator_qwen = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [9]:
messages_qwen = [
    {"role": "user", "content": "讲一个关于鸡的笑话"}
]

output_qwen = generator_qwen(messages_qwen)
print(output_qwen[0]["generated_text"])

在一片广阔的田野上，一只小鸡正在悠闲地觅食。突然，它发现前方有一只大公鸡正站在它的面前，准备抢食。小鸡感到非常尴尬和困惑，但它并没有放弃，而是用一种机智的方式回应了大公鸡。

“大公鸡先生，请问您是想吃我吗？”小鸡礼貌地说道，“我是来寻找食物的。”

大公鸡听了这话后，立刻明白了小鸡的意思，并没有表现出任何的愤怒或不满。相反，它微笑着对小鸡说：“谢谢你，小家伙。你真是个聪明的小朋友。”然后，它继续在自己的领地上享受着美食。

这个故事告诉我们，有时候，我们不需要过于在意别人的反应，只要保持礼貌和尊重，就能得到他人的理解和接纳。
